In [2]:
from dotenv import load_dotenv
from pydantic_ai.models.openai import OpenAIChatModel

# 1. 加载环境变量
load_dotenv()

# 2. 配置模型 (确保环境变量中有 OPENAI_BASE_URL 和 OPENAI_API_KEY)
model = OpenAIChatModel(
    model_name='qwen-plus',
)

In [1]:
import os
from dotenv import load_dotenv
import dashscope
from dashscope import TextEmbedding
#调用阿里的embedding
load_dotenv()

dashscope.api_key = os.getenv("DASHSCOPE_API_KEY")

response = TextEmbedding.call(
    model="text-embedding-v2",
    input="这是一个测试文本"
)

embedding = response["output"]["embeddings"][0]["embedding"]
print(response)
print(len(embedding))  # 向量维度
print(embedding[:10])  # 前10个值

{"status_code": 200, "request_id": "c7058456-131d-9206-b99e-9510c8501754", "code": "", "message": "", "output": {"embeddings": [{"embedding": [-0.010916254100635766, 0.04517456242414186, 0.015137392049921298, -0.0023708957769786707, 0.06189138688555079, -0.03424433104552833, -0.02111966702440543, -0.012705345681789899, -0.028737283522288264, 0.004577558537718583, 0.033321830698995726, -0.019414439111117898, 0.03516683139206093, 0.001998750750820633, 0.022014212814982496, 0.018058643147274532, -0.005825030597234258, -0.005618865747062199, -0.02887705630206593, 0.014871823768343732, -0.0184080750967187, -0.03553024061948286, -0.006373638757861599, 0.009993753754103165, 0.005751649887850982, -0.023985009009847597, -0.01877148432414063, 0.0003658115720743619, 0.007240229992483132, 0.009644321804658998, 0.009742162750503365, 0.0328745578037072, -0.0013068754909211831, 0.028010465067444396, 0.014480459984966264, 0.01872955249020733, 0.023439895168714697, 0.009986765115114283, 0.0037004843446

In [3]:
import os
from pathlib import Path
from scipy.spatial.distance import cosine
from pydantic_ai import Embedder
from pydantic_ai.embeddings.sentence_transformers import SentenceTransformerEmbeddingModel

# ---------------------------------------------------------
# 1. 配置本地的embedding
# ---------------------------------------------------------
# 确保这个路径指向你 git clone 下载的那个文件夹
model_path = os.path.abspath('./bge-small-zh-v1.5')

# 加载模型到内存中（第一次运行会稍慢，因为要读取 100MB 文件）
model = SentenceTransformerEmbeddingModel(model=model_path)
embedder = Embedder(model=model)
# ---------------------------------------------------------
# 3. 定义学习演示函数
# ---------------------------------------------------------
async def run_embedding_study():
    print("🚀 正在启动本地文本嵌入学习...")

    # 我们准备三句话来测试语义相似度
    sentences = [
        "我今天中午吃的牛肉面",  # 目标句
        "西红柿非常的美味",  # 语义相关
        "就该拌42号混凝土"   # 完全无关
    ]

    # 获取向量 (Embeddings)
    # embed_query 会返回一个包含向量列表的对象
    results = []
    for s in sentences:
        res = await embedder.embed_query(s)
        results.append(res.embeddings[0])
        print(f"✅ 已完成句子嵌入: '{s}' (维度: {len(res.embeddings[0])})")

    # ---------------------------------------------------------
    # 4. 数学原理演示：计算余弦相似度
    # ---------------------------------------------------------
    # 相似度 = 1 - 余弦距离。结果越接近 1，表示两句话意思越接近。
    sim_related = 1 - cosine(results[0], results[1])
    sim_unrelated = 1 - cosine(results[0], results[2])

    print("\n--- 语义对比报告 ---")
    print(f"句子 A: {sentences[0]}")
    print(f"句子 B: {sentences[1]}")
    print(f"📊 A 与 B 的相似度: {sim_related:.4f} (语义相关，得分高)")

    print("-" * 20)
    print(f"句子 A: {sentences[0]}")
    print(f"句子 C: {sentences[2]}")
    print(f"📊 A 与 C 的相似度: {sim_unrelated:.4f} (风马牛不相及，得分低)")

# ---------------------------------------------------------
# 5. 执行 (Jupyter 环境专用直接 await)
# ---------------------------------------------------------
await run_embedding_study()

🚀 正在启动本地文本嵌入学习...


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 4432.47it/s]
BertModel LOAD REPORT from: E:\learn\projectNewENV\bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 已完成句子嵌入: '我今天中午吃的牛肉面' (维度: 512)
✅ 已完成句子嵌入: '西红柿非常的美味' (维度: 512)
✅ 已完成句子嵌入: '就该拌42号混凝土' (维度: 512)

--- 语义对比报告 ---
句子 A: 我今天中午吃的牛肉面
句子 B: 西红柿非常的美味
📊 A 与 B 的相似度: 0.4716 (语义相关，得分高)
--------------------
句子 A: 我今天中午吃的牛肉面
句子 C: 就该拌42号混凝土
📊 A 与 C 的相似度: 0.4256 (风马牛不相及，得分低)
